In [1]:
# Librerias
import pandas as pd
import numpy as np
import random
import time

In [2]:
# Carga de los datos

RUTA_CR = "../../datos/CR.txt"

COLUMNAS = [
    "geonameid", "nombre", "nombre_ascii", "nombres_alternos", "latitud", "longitud",
    "clase", "codigo", "pais", "cc2", "admin1", "admin2", "admin3", "admin4",
    "poblacion", "elevacion", "dem", "zona_horaria", "fecha_modificacion",
]

def leer_geonames(ruta=RUTA_CR):
    df = pd.read_csv(
        ruta,
        sep="\t",
        header=None,
        names=COLUMNAS,
        dtype={"admin1": str, "admin2": str, "admin3": str, "admin4": str, "cc2": str},
    )
    return df

df = leer_geonames()

In [3]:
# filtrado
pob_min = 1
N_LOCALIDADES = 50

df_filtrado = df[df["clase"] == "P"]
df_filtrado = df_filtrado[~df_filtrado["codigo"].isin(["PPLQ", "PPLW", "PPLH"])]
df_filtrado = df_filtrado[df_filtrado["poblacion"] >= pob_min]

# ordenar primero (desempate por geonameid) para conservar, entre repetidos, el más poblado
df_filtrado = df_filtrado.sort_values(["poblacion", "geonameid"], ascending=[False, True])

# coordenadas repetidas = iguales al redondear a 3 decimales (~100 m)
df_filtrado = df_filtrado[~df_filtrado[["latitud", "longitud"]].round(3).duplicated()]

df_filtrado = df_filtrado.head(N_LOCALIDADES).reset_index(drop=True)
df_filtrado.head()

,geonameid,nombre,nombre_ascii,nombres_alternos,latitud,longitud,clase,codigo,pais,cc2,admin1,admin2,admin3,admin4,poblacion,elevacion,dem,zona_horaria,fecha_modificacion
0,3621849,San José,San Jose,"SJO,San Chose,San Chosė,San Hose,San Jose,San ...",9.93388,-84.08489,P,PPLC,CR,NaN,08,101,10103,NaN,335007,NaN,1133,America/Costa_Rica,2025-05-15
1,3622247,Limón,Limon,"Ciudad de Limon,Ciudad de Limón,LIO,Limon,Limo...",9.99074,-83.03596,P,PPLA,CR,NaN,06,701,70101,NaN,63081,NaN,29,America/Costa_Rica,2026-04-13
2,3621911,San Francisco,San Francisco,San Francisco,9.99380,-84.12742,P,PPLX,CR,NaN,04,401,40103,NaN,55923,NaN,1131,America/Costa_Rica,2021-11-11
3,3624955,Alajuela,Alajuela,"Alachouela,Alachuela,Alajuela,Alakhuehla,Alaxu...",10.01723,-84.21275,P,PPLA,CR,NaN,01,201,20101,NaN,47494,NaN,955,America/Costa_Rica,2026-04-13
4,3623076,Liberia,Liberia,"LIR,Liberia,Liberii,Liberija,Liberiya,Liberya,...",10.63517,-85.43779,P,PPLA,CR,NaN,03,501,50101,NaN,45380,NaN,148,America/Costa_Rica,2026-04-13


In [4]:
# Análisis descriptivo de los datos

# estructura del archivo completo
df.info()
display(df["clase"].value_counts())

# resumen de las localidades seleccionadas
display(df_filtrado[["poblacion", "latitud", "longitud"]].describe())
display(df_filtrado["codigo"].value_counts())
display(df_filtrado["admin1"].value_counts())

# cadena de filtrado y proporción de población cero
print("Registros totales / lugares poblados / seleccionadas:",
      len(df), len(df[df["clase"] == "P"]), len(df_filtrado))
print("Proporción de lugares poblados con población 0:",
      round((df[df["clase"] == "P"]["poblacion"] == 0).mean(), 3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21426 entries, 0 to 21425
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   geonameid           21426 non-null  int64  
 1   nombre              21426 non-null  object 
 2   nombre_ascii        21426 non-null  object 
 3   nombres_alternos    18538 non-null  object 
 4   latitud             21426 non-null  float64
 5   longitud            21426 non-null  float64
 6   clase               21426 non-null  object 
 7   codigo              21426 non-null  object 
 8   pais                21426 non-null  object 
 9   cc2                 71 non-null     object 
 10  admin1              21364 non-null  object 
 11  admin2              5079 non-null   object 
 12  admin3              4923 non-null   object 
 13  admin4              1 non-null      object 
 14  poblacion           21426 non-null  int64  
 15  elevacion           77 non-null     float64
 16  dem 

clase
H    6801
P    5900
S    4401
T    2908
L     798
A     565
V      44
R       6
U       3
Name: count, dtype: int64

,poblacion,latitud,longitud
count,50.000000,50.000000,50.000000
mean,31049.780000,9.980072,-84.197789
std,45286.994488,0.253129,0.491095
min,11680.000000,9.172980,-85.585100
25%,15495.500000,9.903132,-84.227330
50%,25197.500000,9.947660,-84.084635
75%,28923.000000,9.997938,-84.032588
max,335007.000000,10.951920,-83.035960


codigo
PPL      33
PPLA      6
PPLX      6
PPLA3     2
PPLC      1
PPLA2     1
PPLA4     1
Name: count, dtype: int64

admin1
08    24
04     5
01     5
02     5
03     4
07     4
06     3
Name: count, dtype: int64

Registros totales / lugares poblados / seleccionadas: 21426 5900 50
Proporción de lugares poblados con población 0: 0.978


In [5]:
# Coordenadas planas (km) y matriz de distancias euclidianas

R_TIERRA = 6371.0  # radio medio de la Tierra (km)

lat = np.radians(df_filtrado["latitud"].to_numpy())
lon = np.radians(df_filtrado["longitud"].to_numpy())
lat0, lon0 = lat.mean(), lon.mean()  # punto de referencia: centro del conjunto

# Proyección equirectangular
x = R_TIERRA * np.cos(lat0) * (lon - lon0)
y = R_TIERRA * (lat - lat0)
coords = np.column_stack((x, y))
df_filtrado["x_km"] = x
df_filtrado["y_km"] = y

# Matriz de distancias euclidianas (n x n)
diferencias = coords[:, None, :] - coords[None, :, :]
matriz_distancias = np.sqrt((diferencias ** 2).sum(axis=2))
n = len(df_filtrado)

In [6]:
# Tabla de localidades seleccionadas y exportación para MATLAB

PROVINCIAS = {"01": "Alajuela", "02": "Cartago", "03": "Guanacaste", "04": "Heredia",
              "06": "Limón", "07": "Puntarenas", "08": "San José"}

tabla = pd.DataFrame({
    "Índice": df_filtrado.index,
    "Nombre": df_filtrado["nombre"],
    "Provincia": df_filtrado["admin1"].map(PROVINCIAS),
    "Código": df_filtrado["codigo"],
    "Población": df_filtrado["poblacion"],
    "Latitud": df_filtrado["latitud"].round(3),
    "Longitud": df_filtrado["longitud"].round(3),
    "x (km)": df_filtrado["x_km"].round(1),
    "y (km)": df_filtrado["y_km"].round(1),
})

# Tabla en formato Markdown
print(tabla.to_markdown(index=False).replace(".", ","))

# Exportar para MATLAB
df_filtrado[["x_km", "y_km", "poblacion"]].to_csv("../../datos/localidades_seleccionadas.csv", index_label="indice")

|   Índice | Nombre            | Provincia   | Código   |   Población |   Latitud |   Longitud |   x (km) |   y (km) |
|---------:|:------------------|:------------|:---------|------------:|----------:|-----------:|---------:|---------:|
|        0 | San José          | San José    | PPLC     |      335007 |     9,934 |    -84,085 |     12,4 |     -5,1 |
|        1 | Limón             | Limón       | PPLA     |       63081 |     9,991 |    -83,036 |    127,2 |      1,2 |
|        2 | San Francisco     | Heredia     | PPLX     |       55923 |     9,994 |    -84,127 |      7,7 |      1,5 |
|        3 | Alajuela          | Alajuela    | PPLA     |       47494 |    10,017 |    -84,213 |     -1,6 |      4,1 |
|        4 | Liberia           | Guanacaste  | PPLA     |       45380 |    10,635 |    -85,438 |   -135,8 |     72,8 |
|        5 | Paraíso           | Cartago     | PPL      |       39702 |     9,838 |    -83,867 |     36,3 |    -15,7 |
|        6 | Puntarenas        | Puntarenas  | P

## Función para el cálculo de la distancia entre rutas para los algoritmos

In [7]:
def longitud_ruta(ruta, matriz_distancias):
    """
    Funcion para calcular la longitud total de una ruta cerrada.

    Entradas:                 ruta ---- Lista con los indices de las ciudades
                 matriz_distancias ---- Matriz de distancias de tamanyo nxn

    Salidas:              longitud ---- Longitud total, incluido el regreso
                                        a la ciudad inicial
    """

    n = len(ruta) # Cantidad de localidades
    longitud = 0 # Contador de la longitud total

    for i in range(n):
        if i == n - 1:
            j = 0  # La ultima ciudad se conecta con la primera, supuesto del problema
        else:
            j = i + 1

        a = ruta[i]
        b = ruta[j]

        longitud = longitud + matriz_distancias[a, b]

    return longitud

## Rutas y longitudes iniciales

In [8]:
# Rutas iniciales

semilla = 2026
n_rutas = 100
ruta_rutas = "../../datos/rutas_iniciales.csv"

def generar_rutas_iniciales(n, n_rutas, semilla):
    """
    Funcion para generar rutas iniciales aleatorias.

    Entradas:         n ---- Cantidad de localidades
                n_rutas ---- Cantidad de rutas a generar
                semilla ---- Semilla del generador aleatorio

    Salidas:      rutas ---- Diccionario {id_ruta: ruta}, con id_ruta de 1 a n_rutas
    """
    generador = random.Random(semilla)  # Generador propio con semilla fija
    rutas = {}

    for id_ruta in range(1, n_rutas + 1):
        ruta = list(range(n))
        generador.shuffle(ruta)  # Orden aleatorio de las localidades
        rutas[id_ruta] = ruta

    return rutas

rutas_iniciales = generar_rutas_iniciales(n, n_rutas, semilla)

# Se guardan las rutas en un archivo CSV
pd.DataFrame.from_dict(rutas_iniciales, orient="index").rename_axis("id_ruta").to_csv(ruta_rutas)

# Longitud inicial de cada ruta
longitudes_iniciales = {
    id_ruta: longitud_ruta(ruta, matriz_distancias)
    for id_ruta, ruta in rutas_iniciales.items()
}

# 2-opt


Según Manthey & van Rhijn (2025), 2-opt es una heurística de búsqueda local que mejora un recorrido mediante la sustitución de dos aristas no adyacentes. El intercambio elimina estas aristas y reconecta sus extremos de manera que se conserve un único recorrido cerrado que visite cada localidad exactamente una vez. Esta modificación implica invertir el orden de un segmento de la ruta.

Considere las aristas $(a,b)$ y $(c,d)$, cuyos extremos aparecen en ese orden en el recorrido. El intercambio las sustituye por $(a,c)$ y $(b,d)$. Si $d(i,j)$ representa la distancia entre las localidades $i$ y $j$, el criterio de aceptación es

$$
d(a,c)+d(b,d)<d(a,b)+d(c,d).
$$

Por tanto, el intercambio se acepta únicamente cuando la suma de las longitudes de las nuevas aristas es estrictamente menor que la de las aristas eliminadas. En el caso de distancias simétricas, invertir el segmento intermedio no modifica su longitud, por lo que esta condición garantiza una reducción de la longitud total del recorrido.

El procedimiento se repite hasta que no existen intercambios 2-opt que reduzcan la longitud. La ruta resultante es un óptimo local respecto de estos movimientos, aunque no necesariamente es un óptimo global (Manthey & van Rhijn, 2025).


In [9]:
def dos_opt(ruta_inicial, matriz_distancias):
    """
    Funcion para implementar el algoritmo 2-opt.

    Entradas:         ruta_inicial ---- Lista con los indices de las ciudades
                 matriz_distancias ---- Matriz de distancias de tamanyo nxn

    Salidas:            ruta_copia ---- Lista con la ruta optimizada
    """
    n = len(ruta_inicial) # Cantidad de localidades
    ruta_copia = ruta_inicial.copy() # Esto es para evitar cambiar la ruta original de referencia
    mejorada = True # Inicialización de variable para el ciclo
    iteraciones = 0

    while mejorada == True:
        mejorada = False
        for i in range(n - 1):
            for j in range(i + 2, n): # La idea es que las dos conexiones no compartan una ciudad
                if i == 0 and j == n - 1: # Se excluye la primera conexión
                    continue
                iteraciones += 1
                if j == n - 1:
                    k = 0
                else:
                    k = j + 1

                # Fijamos las localidades de las nuevas conexiones
                a = ruta_copia[i]
                b = ruta_copia[i + 1]
                c = ruta_copia[j]
                d = ruta_copia[k]

                # Encontramos distancias actuales para hacer la comparación
                dist_vieja1 = matriz_distancias[a, b]
                dist_vieja2 = matriz_distancias[c, d]

                # Longitudes de las aristas nuevas para comparar luego
                dist_nueva1 = matriz_distancias[a, c]
                dist_nueva2 = matriz_distancias[b, d]

                # Condicional de aceptación de la nueva ruta
                if dist_nueva1 + dist_nueva2 < dist_vieja1 + dist_vieja2:

                    ruta_nueva = (
                        ruta_copia[:i + 1]
                        + ruta_copia[i + 1:j + 1][::-1]
                        + ruta_copia[j + 1:]
                    )

                    ruta_copia = ruta_nueva
                    mejorada = True

                    break  

            if mejorada == True:
                break  

    return ruta_copia, iteraciones
    

# X-opt 
De acuerdo con Manthey & van Rhijn (2025), el algoritmo X-opt corresponde a una variante del algoritmo 2-opt, en la que el reemplazo de dos aristas solo está permitido si dichas aristas se cruzan entre sí. Lo anterior solo está bien definido para distancias euclidianas en $\mathbb{R}^{2}$. Considere $L(x, y)$ como el segmento de recta entre $x$ e $y$, en vías de facilitar la notación si $e = \{x, y\}$ con $x \neq y$, entonces se escribe $L(e) = L(x, y)$.
Se dice que un conjunto $T$ de aristas presenta cruces si existen $e, f \in T$ que no comparten ningún extremo y que cumplen $L(e) \cap L(f) \neq \emptyset$. Por el contrario, $T$ no presenta cruces si para todo par de aristas no adyacentes $e, f \in T$, se cumple que $L(e) \cap L(f) = \emptyset$. Bajo X-opt un óptimo local es precisamente un recorrido sin cruces entre sus aristas, donde se entiende a un óptimo local como un recorrido que no puede mejorarse mediante ninguna de las modificaciones permitidas por el algoritmo. Por otro lado, con el objetivo de evitar contradicciones en lo que concierne a la definición de cruce, se asume que no es posible que más de un punto se sitúe en una misma posición.

De manera general, sea $T$ un recorrido a través de un conjunto de puntos $\mathcal{X} ⊆ \mathbb{R}^{2}$. Si $T$ presenta cruces, el modo de operación del algoritmo radica en identificar dos aristas no adyacentes $e, f \in T$ que se intersecten, y sustituirlas por nuevas aristas que permitan obtener un recorrido válido con una menor longitud total. Este procedimiento se repite mientras existan cruces en el recorrido, hasta obtener uno sin intersecciones entre aristas no adyacentes. 

In [10]:
# Se construyen las funciones requeridas para determinar si dos aristas se cruzan

def orientacion(a, b, c):
    """
    Funcion para determinar la orientacion de tres puntos en el plano.

    Entradas:         a ---- Punto que pertenece al plano
                      b ---- Punto que pertenece al plano
                      c ---- Punto que pertenece al plano

    Salida: orientacion ---- Valor que describe la orientacion de los tres puntos en el plano: 
                             {0: los puntos están alineados,
                              1: giro antihorario (hacia la izquierda),
                             -1: giro horario (hacia la derecha)}
    """
    # Se inicializa la variable de orientacion 
    orientacion = 1
    # Se calcula el determinante de los vectores AB y AC. 
    det = ((b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0]))
    
    # Se define una tolerancia
    tolerancia = 1e-9

    if abs(det) < tolerancia:
        orientacion = 0
    elif det > 0:
        orientacion =  1
    else:
        orientacion =  -1

    return orientacion


def punto_contenido_segmento(a, b, p):
    """
    Funcion para determinar si un punto P se encuentra dentro del segmento AB, suponiendo que los tres puntos están alineados.

    Entradas:         a ---- Primer extremo del segmento
                      b ---- Segundo extremo del segmento
                      P ---- Punto cuya pertenencia al segmento se quiere comprobar


    Salida:   contenido ---- Indica si el punto esta contenido en el segmento
    """
    # Se define la tolerancia 
    tolerancia = 1e-9

    # Se inicializa la variable
    contenido = False
    
    # Se verifica si las coordenadas de P estan dentro de los limites del segmento AB
    if (min(a[0], b[0]) - tolerancia <= p[0] <= max(a[0], b[0]) + tolerancia
       and min(a[1], b[1]) - tolerancia <= p[1] <= max(a[1], b[1]) + tolerancia): 
       contenido = True
    
    return contenido 

def aristas_se_cruzan(a, b, c, d):
    """
    Funcion para determinar si los segmentos AB y CD se intersectan

    Entradas:         a ---- Primer extremo del primer segmento
                      b ---- Segundo extremo del primer segmento
                      c ---- Primer extremo del segundo segmento
                      d ---- Segundo extremo del segundo segmento

    Salida:   se_cruzan ---- Indica si los segmentos se cruzan entre si

    """
    # Se verifica como se situan los extremos de un segmento respecto a un extremo del otro segmento
    # Para que dos segmentos se crucen en su interior, los extremos de cada segmento deben quedar en lados opuestos de la recta 
    # que contiene al otro segmento o bien, los segmentos deben estar alineados y uno de los extremos de uno de los segmentos debe estar contenido 
    # en el otro segmento 
    o_1 = orientacion(a, b, c)
    o_2 = orientacion(a, b, d)
    o_3 = orientacion(c, d, a)
    o_4 = orientacion(c, d, b)

    # Se inicializa la variable 
    se_cruzan = False

    # Los extremos quedan en lados opuestos:
    if o_1 * o_2 < 0 and o_3 * o_4 < 0:
         se_cruzan = True

    # Segmentos alineados o que se tocan:
    if o_1 == 0 and punto_contenido_segmento(a, b, c):
        se_cruzan = True

    if o_2 == 0 and punto_contenido_segmento(a, b, d):
        se_cruzan = True

    if o_3 == 0 and punto_contenido_segmento(c, d, a):
        se_cruzan = True

    if o_4 == 0 and punto_contenido_segmento(c, d, b):
        se_cruzan = True

    return se_cruzan

In [11]:
def x_opt(ruta_inicial, matriz_distancias, coords): 
    """
    Funcion para implementar el algoritmo X-opt que elimina cruces entre aristas.

    Entradas:         ruta_inicial ---- Lista con los indices de las ciudades
                 matriz_distancias ---- Matriz de distancias de tamanyo nxn
                            coords ---- Matriz con las coordenadas (x, y) de cada localidad

    Salidas:            ruta_copia ---- Lista con la nueva ruta optimizada
    """
     
    n = len(ruta_inicial)
    ruta_inicial_copia = ruta_inicial.copy()
    mejorar = True
    iteraciones = 0
    
    while mejorar == True:
          mejorar = False
          # Se recorren las posibles parejas de aristas mediante dos ciclos for
          # Este primer ciclo recorre las posiciones de las aristas de la ruta mediante el indice i
          for i in range(n - 1):
            # Este segundo ciclo busca una segunda arista para compararla con la arista seleccionada mediante el indice i
            for j in range(i + 2, n): # Empieza en i+2 para que las aristas no sean adyacentes 
                # La primera y la última arista comparten una localidad, por lo que no se consideran: son adyacentes
                if i == 0 and j == n - 1:
                    continue # Se omite la pareja actual y se pasa a la siguiente
                iteraciones += 1
                if j == n - 1:
                    k = 0
                else:
                    k = j + 1
                # Se identifican los extremos de las aristas
                a = ruta_inicial_copia[i]
                b = ruta_inicial_copia[i + 1]
                c = ruta_inicial_copia[j]
                d = ruta_inicial_copia[k]

                # Se verifica si las aristas se cruzan, solo se permiten cambios entre aristas que se cruzan
                if not aristas_se_cruzan(coords[a], coords[b], coords[c], coords[d]):
                    # Si las aristas no se cruzan, entonces se omite la pareja actual y se pasa a la siguiente
                    continue

                # Longitud de las aristas actuales que forman parte de la ruta inicial
                dist_vieja = (matriz_distancias[a, b] + matriz_distancias[c, d])

                # Longitud de las nuevas aristas
                dist_nueva = (matriz_distancias[a, c] + matriz_distancias[b, d])

                # Se acepta el cambio solo si mejora la ruta
                if dist_nueva < dist_vieja:
                    # Se invierte el tramo comprendido entre las dos aristas seleccionadas
                    ruta_nueva = (
                        ruta_inicial_copia[:i + 1]
                        + ruta_inicial_copia[i + 1:j + 1][::-1]
                        + ruta_inicial_copia[j + 1:]
                    )
                    ruta_inicial_copia = ruta_nueva
                    mejorar = True
                    break # Salir del for j

            # Si se encontro una mejora, se reinicia la busqueda.
            if mejorar == True:
                break # Salir del for i

    return ruta_inicial_copia, iteraciones

# Y-opt

El algoritmo Y-opt es una variante del algoritmo 2-opt. Como se vio anteriormente, el algoritmo 2-opt acepta un intercambio de aristas si la suma de las longitudes de las dos aristas nuevas es menor que la suma de las dos aristas viejas; esto se hace comparando únicamente las aristas que se modificaron, ya que el resto de la ruta permanece igual. Por otro lado, el algoritmo Y-opt impone una condición de cambio más estricta, donde, para aceptar un cambio en la ruta, la arista más larga de las nuevas debe ser más corta que la arista más larga de las viejas, y, además, la arista más corta de las nuevas debe ser más corta que la arista más corta de las viejas. Esto hace que el algoritmo Y-opt sea más restrictivo en su condición de aceptación de nuevas rutas, ya que no solo compara la suma total, sino el tamaño individual de cada arista.

La principal ventaja del algoritmo Y-opt es que el número de iteraciones para una ruta de $n$ vértices y $m$ aristas está acotado por $(m-1)n$ (Manthey & van Rhijn, 2025), resultando en un tiempo de ejecución polinomial. En comparación, el algoritmo 2-opt puede necesitar un tiempo exponencial en los peores casos; de esta forma, el algoritmo Y-opt compensa su condición de aceptación con el tiempo que necesitará para ejecutarse. A pesar de que el algoritmo Y-opt sea más estricto, los resultados teóricos muestran que este algoritmo tiene resultados similares a los del 2-opt, donde se llega a rutas ligeramente más largas con el algoritmo Y-opt pero en un tiempo de ejecución acotado en comparación al 2-opt (Manthey & van Rhijn, 2025).

In [22]:
# Funcion Y-opt

def y_opt(ruta_inicial, matriz_distancias):
    """
    Función para implementar el algoritmo Y-opt
    Entradas :           ruta_inicial ---- Lista con los indices de las ciudades 
                    matriz_distancias ---- Matriz de distancias de tamanyo nxn
                    
    Salidas :             ruta_nueva ---- Lista con la nueva ruta optimizada
    """
    
    n = len(ruta_inicial)
    ruta_copia = ruta_inicial.copy() # Copy para evitar sobreescribir la lista original
    mejorada = True 
    iteraciones = 0 
    
    while mejorada == True: # Hacer el while hasta que la ruta no se pueda mejorar mas
        mejorada = False # Cambiar la condicion de si se mejoro o no la ruta (Se va a cambiar más adelante)
        for i in range(n-1):
            for j in range(i + 2, n): # i + 2 para evitar usar la misma arista en las comparaciones
                if i == 0 and j == n - 1: # Se excluyen la primera y ultima arista
                    continue

                iteraciones += 1

                if (j == n-1): # Se llega a la última ciudad, la cual debe conectar con la primera siempre
                    k = 0
                else:
                    k = j + 1 # Si no estamos en la última ciudad, la ciudad actual se conecta con la siguiente

                # Las 4 ciudades consecutivas son: (Ordenadas segun la ruta inicial)
                a = ruta_copia[i]
                b = ruta_copia[i+1]
                c = ruta_copia[j]
                d = ruta_copia[k]
                
                # Largo de las aristas viejas:
                dist_vieja1 = matriz_distancias[a,b]
                dist_vieja2 = matriz_distancias[c,d]
                
                # Largo de las aristas nuevas:
                dist_nueva1 = matriz_distancias[a,c]
                dist_nueva2 = matriz_distancias[b,d]
                
                # Crear lista con las distancias nuevas
                
                dist_viejas = np.array([dist_vieja1,dist_vieja2])
                dist_nuevas = np.array([dist_nueva1,dist_nueva2])
                
                if (np.max(dist_nuevas) < np.max(dist_viejas)) and (np.min(dist_nuevas) < np.min(dist_viejas)): # Si esto se cumple se puede cambiar la ruta
                    ruta_nueva = ruta_copia[:i+1] + ruta_copia[i+1:j+1][::-1] + ruta_copia[j+1:] # Los + son para concatenar las listas, hay que recordar que en Python las listas empiezan en 0 y se incluye el extremo izquierdo pero no el derecho cuando se extrae una sublista; el [::-1] es para darle vuelta al segmento que está entre las aristas cambiadas
                    ruta_copia = ruta_nueva
                    mejorada = True
                    break # Salir del for de j
            if mejorada == True:
                break  # Salir del for de i
          
    return ruta_copia, iteraciones
            
            
    

In [23]:
# Método para ejecutar los tres algoritmos

def ejecutar_algoritmos(matriz_distancias, n):
    """
    Metodo para ejecutar los algoritmos 2-opt, X-opt y Y-opt
    Entradas:         matriz_distancias  ---- Matriz de tamanyo nxn que posee las distancias entre las ciudades
                                      n  ---- Numero de ciudades
    Salidas:                ruta_inicial ---- Ruta inicial creada aleatoriamente
                              ruta_2_opt ---- Ruta optimizada con el algoritmo de 2-opt
                              ruta_X_opt ---- Ruta optimizada con el algoritmo de X-opt
                              ruta_Y_opt ---- Ruta optimizada con el algoritmo de Y-opt                       
    """
    ruta_inicial = list(range(n)) # Crear una lista de n elementos
    random.shuffle(ruta_inicial) # Ordenar la lista de la ruta de forma aleatoria 
    
    # Aplicar los tres algoritmos
    ruta_dos_opt = dos_opt(ruta_inicial, matriz_distancias)
    ruta_x_opt = x_opt(ruta_inicial, matriz_distancias, coords)
    ruta_y_opt = y_opt(ruta_inicial, matriz_distancias)
    
    return ruta_inicial, ruta_dos_opt, ruta_x_opt, ruta_y_opt
    


# Medidas a Recopilar en los experimentos

Para evaluar y poder comparar el rendimiento de los tres algoritmos, 2-opt, X-opt y Y-opt, se utilizarán las siguientes medidas:

Longitud de la ruta optimizada: Es la longitud final medida en kilómetros de cada ruta optimizada según el algoritmo; esta medida incluye todo el recorrido realizado y el regreso desde la última visitada hasta el punto de partida inicial. De esta forma, se puede comparar qué tan grande o pequeña es la ruta de un algoritmo con respecto a las demás. 

Tiempo de ejecución: Es el tiempo medido en segundos, de lo que tardó cada algoritmo en encontrar su solución. Esto permitirá comparar la rapidez de cada algoritmo para encontrar una solución. 

Número de intercambios aceptados: Es el número de veces que cada algoritmo aceptó algún cambio en la ruta. Esta medida permitirá ver qué tanto aceptó o rechazó cambios un algoritmo, permitiendo así, ver qué tan estricto es cada algoritmo con la aceptación de un cambio. 

Ruta final obtenida: Es una lista los índices que representan el orden en que se visitó cada ciudad. Esto permitirá realizar los gráficos y comparar las rutas creadas por cada algoritmo.

Porcentaje de mejora con respecto a la ruta inicial: Esta medida se calculará como $$((longitud inicial - longitud final) / longitud inicial) \times 100 $$; esta medida permite ver cuánto mejoró la solución de cada algoritmo con respecto a la ruta inicial creada aleatoriamente.

Las métricas antes mencionadas permitirán realizar la comparación entre algoritmos, además de la visualización de cada ruta creada. Es importante destacar que, para lograr realizar esta comparación, los tres algoritmos utilizarán la misma ruta inicial y la misma matriz de distancias.

# Prueba preliminar con 15 localidades

In [24]:
# Y opt

ruta_prueba = list(range(15))
random.shuffle(ruta_prueba)

inicio = time.time() 
ruta_opt,iteraciones = y_opt(ruta_prueba, matriz_distancias)
fin = time.time() 
tiempo_ejecucion = fin - inicio

print("Ruta antes:", ruta_prueba)
print("Longitud antes:", longitud_ruta(ruta_prueba, matriz_distancias))
print("Ruta después:", ruta_opt)
print("Longitud después:", longitud_ruta(ruta_opt, matriz_distancias))
print("Número de Iteraciones:", iteraciones)
print("Tiempo de ejecución:", tiempo_ejecucion ,"segundos") 

Ruta antes: [0, 3, 9, 13, 14, 5, 10, 2, 6, 1, 11, 4, 8, 12, 7]
Longitud antes: 1317.575638189174
Ruta después: [0, 2, 3, 6, 4, 10, 9, 11, 14, 8, 13, 5, 1, 12, 7]
Longitud después: 730.0128890539347
Número de Iteraciones: 377
Tiempo de ejecución: 0.012012958526611328 segundos


In [25]:
# 2 opt
inicio = time.time() 
ruta_opt,iteraciones = dos_opt(ruta_prueba, matriz_distancias)
fin = time.time() 
tiempo_ejecucion = fin - inicio

print("Ruta antes:", ruta_prueba)
print("Longitud antes:", longitud_ruta(ruta_prueba, matriz_distancias))
print("Ruta después:", ruta_opt)
print("Longitud después:", longitud_ruta(ruta_opt, matriz_distancias))
print("Número de Iteraciones:", iteraciones)
print("Tiempo de ejecución:", tiempo_ejecucion ,"segundos") 

Ruta antes: [0, 3, 9, 13, 14, 5, 10, 2, 6, 1, 11, 4, 8, 12, 7]
Longitud antes: 1317.575638189174
Ruta después: [0, 2, 3, 10, 4, 6, 7, 1, 12, 5, 13, 8, 11, 9, 14]
Longitud después: 686.105841288418
Número de Iteraciones: 635
Tiempo de ejecución: 0.0 segundos


In [26]:
# X opt
inicio = time.time() 
ruta_opt,iteraciones = x_opt(ruta_prueba, matriz_distancias, coords)
fin = time.time() 
tiempo_ejecucion = fin - inicio

print("Ruta antes:", ruta_prueba)
print("Longitud antes:", longitud_ruta(ruta_prueba, matriz_distancias))
print("Ruta después:", ruta_opt)
print("Longitud después:", longitud_ruta(ruta_opt, matriz_distancias))
print("Número de Iteraciones:", iteraciones)
print("Tiempo de ejecución:", tiempo_ejecucion ,"segundos") 

Ruta antes: [0, 3, 9, 13, 14, 5, 10, 2, 6, 1, 11, 4, 8, 12, 7]
Longitud antes: 1317.575638189174
Ruta después: [0, 2, 3, 6, 4, 10, 1, 12, 11, 9, 5, 7, 8, 14, 13]
Longitud después: 766.0826284998277
Número de Iteraciones: 654
Tiempo de ejecución: 0.006038665771484375 segundos


# Ejecución de los algoritmos con 100 rutas aleatorias de 50 localidades

In [27]:
# Ejecución del algoritmo y-opt

resultados_y_opt = [] # Lista vacía para guardar los resultados del algoritmo y_opt en cada iteracion


for id_ruta, ruta_inicial in rutas_iniciales.items(): # Recorrer cada elemento del diccionario
    

    inicio = time.time() 
    ruta_y_opt,iteraciones = y_opt(ruta_inicial, matriz_distancias)
    fin = time.time() 
    
    tiempo = fin - inicio
    
    longitud_inicial = longitud_ruta(ruta_inicial,matriz_distancias)
    longitud_final = longitud_ruta(ruta_y_opt, matriz_distancias)
    mejora = ((longitud_inicial - longitud_final)/longitud_inicial) * 100
    
    resultados_y_opt.append({ # Guardar un diccionario con cada resultado en cada entrada de la lista
        "id_ruta" : id_ruta,
        "ruta_inicial" : ruta_inicial,
        "longitud_inicial" : longitud_inicial,
        "ruta_final": ruta_y_opt,
        "longitud_final" : longitud_final,
        "tiempo_ejecucion" : tiempo,
        "intercambios_aceptados" : iteraciones,
        "porcentaje_mejora" : mejora
    })
    
    
df_y_opt = pd.DataFrame(resultados_y_opt) # Convertir la lista a Data Frame
display(df_y_opt.head())

,id_ruta,ruta_inicial,longitud_inicial,ruta_final,longitud_final,tiempo_ejecucion,intercambios_aceptados,porcentaje_mejora
0,1,"[37, 23, 29, 24, 22, 13, 16, 42, 30, 4, 27, 19...",3265.021018,"[37, 39, 13, 26, 8, 14, 22, 30, 35, 20, 29, 46...",986.319409,0.210606,40644,69.791330
1,2,"[0, 6, 14, 46, 2, 28, 9, 17, 8, 12, 49, 11, 10...",3216.278402,"[0, 20, 35, 30, 22, 9, 11, 17, 43, 44, 41, 34,...",994.634060,0.143939,31466,69.075001
2,3,"[37, 20, 19, 30, 11, 46, 40, 34, 6, 42, 28, 48...",3433.137103,"[37, 39, 13, 26, 8, 14, 22, 30, 20, 35, 0, 42,...",1057.480241,0.178285,40722,69.197844
3,4,"[33, 11, 27, 37, 42, 46, 29, 10, 1, 38, 6, 3, ...",2963.379365,"[33, 1, 49, 7, 12, 5, 18, 34, 26, 13, 39, 37, ...",939.911070,0.190426,40741,68.282459
4,5,"[26, 42, 4, 37, 16, 49, 18, 45, 3, 8, 35, 44, ...",3302.725156,"[26, 13, 39, 37, 23, 42, 25, 0, 30, 35, 20, 9,...",905.039354,0.246970,51946,72.597194


In [31]:
# Ejecución del algoritmo 2-opt

resultados_dos_opt = [] 


for id_ruta, ruta_inicial in rutas_iniciales.items(): 
    

    inicio = time.time() 
    ruta_dos_opt,iteraciones = dos_opt(ruta_inicial, matriz_distancias)
    fin = time.time() 
    
    tiempo = fin - inicio
    
    longitud_inicial = longitud_ruta(ruta_inicial,matriz_distancias)
    longitud_final = longitud_ruta(ruta_dos_opt, matriz_distancias)
    mejora = ((longitud_inicial - longitud_final)/longitud_inicial) * 100
    
    resultados_dos_opt.append({
        "id_ruta" : id_ruta,
        "ruta_inicial" : ruta_inicial,
        "longitud_inicial" : longitud_inicial,
        "ruta_final": ruta_dos_opt,
        "longitud_final" : longitud_final,
        "tiempo_ejecucion" : tiempo,
        "intercambios_aceptados" : iteraciones,
        "porcentaje_mejora" : mejora
    })
    
    
df_dos_opt = pd.DataFrame(resultados_dos_opt) 
display(df_dos_opt.head())

,id_ruta,ruta_inicial,longitud_inicial,ruta_final,longitud_final,tiempo_ejecucion,intercambios_aceptados,porcentaje_mejora
0,1,"[37, 23, 29, 24, 22, 13, 16, 42, 30, 4, 27, 19...",3265.021018,"[37, 39, 13, 26, 34, 18, 5, 7, 49, 1, 33, 32, ...",899.555464,0.123659,72753,72.448708
1,2,"[0, 6, 14, 46, 2, 28, 9, 17, 8, 12, 49, 11, 10...",3216.278402,"[0, 35, 20, 29, 46, 28, 2, 21, 3, 48, 16, 31, ...",902.164443,0.018726,39504,71.950051
2,3,"[37, 20, 19, 30, 11, 46, 40, 34, 6, 42, 28, 48...",3433.137103,"[37, 23, 13, 26, 34, 18, 5, 12, 32, 33, 1, 49,...",891.695986,0.038448,44688,74.026788
3,4,"[33, 11, 27, 37, 42, 46, 29, 10, 1, 38, 6, 3, ...",2963.379365,"[33, 1, 49, 7, 27, 15, 40, 36, 19, 6, 38, 45, ...",882.517778,0.071604,49562,70.219210
4,5,"[26, 42, 4, 37, 16, 49, 18, 45, 3, 8, 35, 44, ...",3302.725156,"[26, 18, 5, 12, 7, 49, 1, 33, 32, 29, 46, 28, ...",897.425407,0.044567,40729,72.827730


# Medidas estadísticas para el análisis de resultados

Se calcula la longitud media, la longitud mediana, el porcentaje medio de mejora y la cantidad de rutas finales más cortas que sus respectivas rutas iniciales.

In [33]:
# Longitudes y porcentajes de mejora de 2-opt

longitudes = df_dos_opt["ruta_final"].apply(
    lambda ruta: longitud_ruta(ruta, matriz_distancias)
)

mejoras = (
    (df_dos_opt["longitud_inicial"] - longitudes)
    / df_dos_opt["longitud_inicial"]
) * 100

print("Resultados 2-opt:")
print("Longitud media:", round(longitudes.mean(), 3), "km")
print("Longitud mediana:", round(longitudes.median(), 3), "km")
print("Porcentaje medio de mejora:", round(mejoras.mean(), 3), "%")
print(
    "Cantidad de rutas más cortas que la inicial:",
    (longitudes < df_dos_opt["longitud_inicial"]).sum()
)

Resultados 2-opt:
Longitud media: 898.311 km
Longitud mediana: 897.133 km
Porcentaje medio de mejora: 71.285 %
Cantidad de rutas más cortas que la inicial: 100


# Qué pasaría si se ejecutara un algoritmo a una ruta ya optimizada por el mismo algoritmo?

Los algoritmos implementados realizan una búsqueda local que modifica la ruta mientras encuentra intercambios que cumplen su criterio de mejora. En particular, 2-opt reemplaza dos aristas por otras dos cuando el cambio reduce estrictamente la longitud total del recorrido. El procedimiento termina cuando ningún intercambio 2-opt permite obtener una ruta más corta, por lo que la solución alcanzada es un óptimo local respecto al vecindario 2-opt (Englert et al., 2014).

En la presente implementación, esta búsqueda se repite mediante un ciclo while. Después de aceptar una mejora, el algoritmo reinicia la exploración sobre la ruta modificada. Por tanto, la función solo finaliza después de examinar todas las parejas válidas de la ruta actual sin aceptar ningún cambio. Al ejecutar nuevamente el mismo algoritmo sobre esa ruta, manteniendo las distancias, las coordenadas y las condiciones de aceptación, se repiten las mismas comprobaciones y la ruta permanece igual. En el ejemplo que se muestra a continuación (para este ejemplo, se utilizó la ruta inicial de la prueba preliminar de las 15 localidades, únicamente para ver el comportamiento y ver si coincide con el esperado respaldado por la teoría), note que la segunda ejecución sí realiza evaluaciones para comprobar que no existen mejoras, pero la ruta no se modifica.

El razonamiento también se aplica a X-opt y Y-opt, considerando sus respectivas restricciones de aceptación. Manthey y van Rhijn (2025) presentan ambos métodos como variantes que restringen los movimientos de 2-opt. Teniendo esto en cuenta, y la forma en la que se implementaron los algoritmos, va a ocurrir lo mismo para Y-opt y X-opt. En consecuencia, que una ruta permanezca igual al repetir un algoritmo indica que ya no admite mejoras bajo su criterio, pero no garantiza que sea la ruta de menor longitud entre todas las posibles, puesto que los algoritmos están diseñados para ser una aproximación.

In [30]:
# Prueba de lo mencionado utilizando el algoritmo 2-opt

ruta_1, _ = dos_opt(ruta_inicial, matriz_distancias)
ruta_2, evaluaciones = dos_opt(ruta_1, matriz_distancias)

print("Prueba con 2-opt")
print("¿La ruta quedó igual?", ruta_1 == ruta_2)
print("Longitud primera ejecución:", longitud_ruta(ruta_1, matriz_distancias))
print("Longitud segunda ejecución:", longitud_ruta(ruta_2, matriz_distancias))
print("Evaluaciones segunda ejecución:", evaluaciones)
print("")

# Prueba de lo mencionado utilizando el algoritmo Y-opt

ruta_1, _ = y_opt(ruta_inicial, matriz_distancias)
ruta_2, evaluaciones = y_opt(ruta_1, matriz_distancias)

print("Prueba con Y-opt")
print("¿La ruta quedó igual?", ruta_1 == ruta_2)
print("Longitud primera ejecución:", longitud_ruta(ruta_1, matriz_distancias))
print("Longitud segunda ejecución:", longitud_ruta(ruta_2, matriz_distancias))
print("Evaluaciones segunda ejecución:", evaluaciones)
print("")

# Prueba de lo mencionado utilizando el algoritmo X-opt

ruta_1, _ = x_opt(ruta_inicial, matriz_distancias, coords)
ruta_2, evaluaciones = x_opt(ruta_1, matriz_distancias, coords)

print("Prueba con X-opt")
print("¿La ruta quedó igual?", ruta_1 == ruta_2)
print("Longitud primera ejecución:", longitud_ruta(ruta_1, matriz_distancias))
print("Longitud segunda ejecución:", longitud_ruta(ruta_2, matriz_distancias))
print("Evaluaciones segunda ejecución:", evaluaciones)

Prueba con 2-opt
¿La ruta quedó igual? True
Longitud primera ejecución: 898.5683778495846
Longitud segunda ejecución: 898.5683778495846
Evaluaciones segunda ejecución: 1175

Prueba con Y-opt
¿La ruta quedó igual? True
Longitud primera ejecución: 1013.7756290532067
Longitud segunda ejecución: 1013.7756290532067
Evaluaciones segunda ejecución: 1175

Prueba con X-opt
¿La ruta quedó igual? True
Longitud primera ejecución: 1296.1361129033303
Longitud segunda ejecución: 1296.1361129033303
Evaluaciones segunda ejecución: 1175


Note que los tres algoritmos realizaron 1175 evaluaciones en la segunda ejecución porque cada uno recibió una ruta que ya no admitía mejoras según su criterio de aceptación. Por ello, completaron una única pasada sin modificarla. Para 50 localidades, existen $50(50-3)/2=1175$ parejas de aristas no adyacentes.

Como el contador se incrementa antes de aplicar los filtros particulares de X-opt y Y-opt, los tres métodos contabilizan la misma cantidad de parejas examinadas. Esta coincidencia no implica que realicen exactamente las mismas operaciones ni que obtengan rutas de igual longitud. En la primera ejecución, el número de evaluaciones puede variar según las mejoras aceptadas y los reinicios de la búsqueda.